# Seismogram Objects
## *Gary L. Pavlis*

## Overview
This is a notebook students in the MsPASS class will want to read and run outside the scheduled class period.   It contains some foundational material on what we call a `Seismogram` object, which encapsulates the concept of the output records of a three-component sensor.  Some key points this tutorial illustrates are:
*  The data array of a `Seismogram` is a 3Xn_samples matrix
*  The object has methods to handle coordinate transformations automatically.   I emphasize they are "methods"  not functions applied to data to transform the data matrix.
*  The object has failsafes you can use to make sure the data are oriented the way you expect them to be.   

## Before Running this Notebook
The assumption is you are running this notebook after starting the database server.   Instructions for doing that on GeoLab are in most of the notebooks in this directory.   I also assume you have run at least the precourse notebooks to assemble the working dataset used for these tutorials.   

## Initializations
First, the stock incantation that most MsPASS workflows should use for initialization.   

In [ ]:
from mspasspy.db.database import Database
import mspasspy.client as msc
mspass_client = msc.Client()
dask_client - mspass_client.get_scheduler()
db = mspass_client.get_database("ES2026")

## Reading and plotting Seismogram objects
First, read a single `Seismogram` object we can experiment with.   I'm going to use nontrivial query to select a single Seismogram I know from experimentation has a visible signal without any filtering.   

In [ ]:
# query is necessary because not all Seismogram data have Ptime set.  This guarantees we find the first one that does have it set
doc = db.wf_Seismogram.find_one({'Ptime' : {'$exists' : 1}})
d = db.read_data(doc,collection='wf_Seismogram')
# shift data so 0 is approximate P wave arrival time
Ptime = d['Ptime']
d = ator(d,Ptime)
plotter = SeismicPlotter(normalize=True)
plotter.change_style('wtva')
plotter.plot(d)

Because a `Seismogram` is a native MsPASS data object the easiest way to plot one is to use the MsPASS plotter.  This box does that to plot the single Seismogram we just loaded.

In [ ]:
from mspasspy.graphics import SeismicPlotter
import matlotlib.pyplot as plt
# instantiate the plotter
plotter = SeismicPlotter(normalize=True)
# The default "style" is not the best for Seismograms - change to wiggle trace variable area (wtva)
plotter.change_style('wtva')

Now plot these data.  We also, however, want to shift the time reference so the P wave arrival time defines time 0.   The `ator` function does here:

In [ ]:
Ptime = d["Ptime"]   # extracts the ANF pick for the P wave first arrival
d = ator(d,Ptime)    # does the time shift
plotter.plot(d)
plt.show()   # plotter uses matplotlib and like all matplotlib graphics a figure should be terminated with this incantation


Noting the components are plotted from the top down are BHE, BHN, and BHZ.   

Note, if you prefer matplotlib graphics you can use them but the syntax it takes a lot more code and some ugly numpy style array gyrations.  For your education, here is the same data plotted with matplotlib subplots.

In [ ]:
fig,ax = subplots(3)
for i in range(3):
    ax[i].plot(d.time_axis(),d.data[:,i])
plt.show()

## Seismogram concepts
The issue we need to address at this point is a unique feature of MsPASS compared to any other framework we know.  The `Seismogram` class was designed to enscapsulate key concepts that define what a three component seismogram is.   There are several features of `Seismogram` that are relevant to this tutorial:
1.  A `Seismogram` object keeps track of the orientation of the components in space relative to the local geographic reference frame at the station.  We discussed this earlier, but we will dig into this a bit deeper here.
2.  A `Seismogram` unifies the generic concept of coordinate transformations to allow simple rotations to coordinates like RTZ to a general coordinate transformation using the `transform` method of `Seismogram` or the [transform function wrapper](http://www.mspass.org/python_api/mspasspy.algorithms.html#mspasspy.algorithms.basic.transform) wrapper function used for parallel processing constructs like those we will use below.  In this tutorial we will be using a particularly useful, but relatively complicated function useful for P wave receiver function processing, called [free_surface_transformation](http://www.mspass.org/python_api/mspasspy.algorithms.html#mspasspy.algorithms.basic.free_surface_transformation).
3.  Unless you apply a singular transformation, the data can always be restored to the standard coordinate reference frame of x1=EW, x2=NS, and x3=Z with the `rotate_to_standard` method of the data object or the [rotate_to_standard wrapper function](http://www.mspass.org/python_api/mspasspy.algorithms.html#mspasspy.algorithms.basic.rotate_to_standard).

To make this clearer we will explore that a bit before proceeding.  A more in depth version of this material can be found [here](https://github.com/mspass-team/mspass_tutorial/blob/master/notebooks/Three-ComponentBasics.ipynb).

First note the output of this simple code blocK:

In [ ]:
print(d.cardinal())
print(d.tmatrix)

Note:
- The `cardinal` method returning False tells you the data are not in standard geographic coordinates (ENZ)
- The output of the `tmatrix` method shows that indeed the transformation is anything but an identity matrix because the order the coordinate is not standard.

WE fix that problem by applying the `rotate_to_standard` function:

In [ ]:
from mspasspy.algorithms.basic import rotate_to_standard
d = rotate_to_standard(d)
print(d.cardinal())
print(d.tmatrix)

Note the transformation matrix is now an identity matrix - the definition of "cardinal" in MsPASS.  Hence, the `cardinal` method now returns True.

MsPASS has a relatively complete set of coordinate transformation operators from simple rotation around Z to the most sophisticated coordinate transformation for P wave data  commonly called the "free surface transformation". The theory for the matrix transformation is in Brian Kennett's (1983) book on layered media theory. We will apply that operator to these data as it is arguably the best way to prepare teleseismic P wave data for receiver function deconvolution.   A more complete tutorial on the transformation operators available for `Seismogram` objects can be found [here](https://github.com/mspass-team/mspass_tutorial/blob/master/notebooks/Three-ComponentBasics.ipynb).

The next code box defines a function that we can apply to `Seismogram` objects to apply the free surface transformation.  Note the operator requires us to know the ray parameter for the P wave being analyzed.  We set those values as Metadata attributes when we created this assembled data set with the notebook you ran prior to the start of the class.

In [ ]:
import math
from mspasspy.ccore.seismic import SlownessVector
from mspasspy.algorithms.basic import free_surface_transformation
from mspasspy.ccore.utility import ErrorSeverity

def apply_FST(d,rayp_key="rayp_P",seaz_key='seaz',vp0=6.0,vs0=3.5):
    """
    Apply free surface transformation operator of Kennett (1983) to an input `Seismogram` 
    object.   Assumes ray parameter and azimuth data are stored as Metadata in the 
    input datum.  If the ray parameter or azimuth key are not defined an error 
    message will be posted and the datum will be killed before returning.  
    :param d:  datum to process
    :type d:  Seismogram
    :param rayp_key:   key to use to extract ray parameter to use to compute the 
    free surface transformation operator.  Note function assumes the ray parameter is
    spherical coordinate form:  R sin(theta)/V.   Default is "rayp_P".
    :param seaz_key:   key to use to extract station to event azimuth. Default is "seaz".
    :param vp0:  surface P wave velocity (km/s) to use for free surface transformation 
    :param vs0:  surface S wave velocity (km/s) to use for free surface transformation.
    """
    if d.is_defined(rayp_key) and d.is_defined(seaz_key):
        rayp = d[rayp_key]
        seaz = d[seaz_key]
        # Some basic seismology here.  rayp is the spherical earth ray parameter
        # R sin(theta)/v.  Free surface transformation needs apparent velocity 
        # at Earth's surface which is sin(theta)/v when R=Re.   Hence the following
        # simple convertion to get apparent slowness at surface  sin(theta)/v
        Re=6378.1
        umag = rayp/Re   # magnitude of slowness vector
        # seaz is back azimuth - slowness vector points in direction of propagation
        # with is 180 degrees away from back azimuth
        az = seaz + 180.0
        # component slowness vector components in local coordinates
        ux = umag * math.sin(az)
        uy = umag * math.cos(az)
        # FST implementation requires this special class called a Slowness Vector
        u = SlownessVector(ux,uy)
        d = free_surface_transformation(d,uvec=u,vp0=vp0,vs0=vs0)
    else:
        d.kill()
        message = "one of required attributes rayp_P or seaz were not defined for this datum"
        d.elog.log_error("apply_FST",message,ErrorSeverity.Invalid)
        
    return d

Run this on d to verify it works and to see the effect.

In [ ]:
d2=apply_FST(d)
plotter.plot(d2)
# plot is from bottom up SH, SV, P